# 🚀 Train M3DUSA vs. CMTF on Kaggle GPU

This notebook provides two training workflows using free Kaggle GPUs (T4 or P100 with 16 GB VRAM):
1. **Part I: PolitiFact Benchmark Reproduction** (M3DUSA Baseline vs. Novel CMTF)
2. **Part II: Large-Scale 29 GB FakeNewsCorpus Streaming** (Full multi-million article training)

### ⚙️ Prerequisites in Kaggle:
1. In the right-hand panel under **Settings**:
   - **Accelerator**: Select **GPU T4 x2** or **GPU P100**
   - **Internet**: Toggle to **Internet on** (required for downloading HuggingFace weights)
2. (For Part II) Under **Input** in the right panel, click **+ Add Input** and attach your uploaded `NEW_DATA` or search for `FakeNewsCorpus`.

In [ ]:
# Step 1: Check GPU & Environment
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Step 2: Clone repository from GitHub
import os
if not os.path.exists("m3dusa-cmtf"):
    !git clone https://github.com/mouryajonnada/m3dusa-cmtf.git

%cd m3dusa-cmtf
!git pull

In [ ]:
# Step 3: Install dependencies
!pip install -q torch-geometric transformers sentence-transformers

In [ ]:
# Step 4: Run smoke test to verify pipeline
!python scripts/smoke_test.py

--- 
## 🏆 Part I: PolitiFact Benchmark Reproduction

In [ ]:
# Step 5: Train Baseline M3DUSA (Late Fusion)
!python scripts/train.py --config experiments/baseline_m3dusa.yaml --device cuda

In [ ]:
# Step 6: Train Novel CMTF (Cross-Modal Transformer Fusion)
!python scripts/train.py --config experiments/cross_modal_fusion.yaml --device cuda

In [ ]:
# Step 7: Generate Benchmark Results & Visualizations
!python scripts/generate_results.py

In [ ]:
# Step 8: Display results comparison table and plots
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv("results/metrics/comparison.csv")
display(df)

print("\n--- Baseline vs. CMTF Performance Metrics ---")
display(Image("results/plots/baseline_vs_cmtf_metrics.png"))

print("\n--- UMAP Latent Embedding Clusters ---")
display(Image("results/plots/umap_embeddings.png"))

---
## 🌊 Part II: Large-Scale 29 GB FakeNewsCorpus Streaming
Train across massive datasets without running out of RAM using the chunk-by-chunk disk streaming engine.

In [ ]:
# Step 9: Locate dataset in /kaggle/input/
import glob
from pathlib import Path

input_candidates = glob.glob("/kaggle/input/**/news_cleaned_2018_02_13*.csv", recursive=True)
if input_candidates:
    data_csv = input_candidates[0]
    print(f"✅ Found dataset: {data_csv}")
else:
    print("⚠️ No CSV found in /kaggle/input/. If you uploaded split parts or a folder, specify path below.")
    data_csv = "NEW_DATA/news_cleaned_2018_02_13.csv"

In [ ]:
# Step 10: Run High-Speed Streaming Training on Kaggle GPU
# With 16 GB VRAM, we can run larger batch size (32) and chunk size (2,048) for ~400+ samples/sec!
!python scripts/train_stream.py \
    --input "$data_csv" \
    --batch-size 32 \
    --grad-accum 2 \
    --chunk-size 2048 \
    --device cuda

In [ ]:
# Step 11: Package trained models for 1-click download
!zip -r /kaggle/working/m3dusa_cmtf_trained_models.zip results/
print("\n✅ Complete! Download 'm3dusa_cmtf_trained_models.zip' from the right-hand Output tab.")